# Creating accounts — Python notebook version

This notebook converts the SAS `PROC RISK` workflow from the GitHub file
`Risk/Creating accounts` into a usable Python backtest.

## What this reproduces
The SAS file defines three portfolio behaviors:

- **ConvexTrading**:  
  sell a stock if its return is negative; buy it with available system cash if its return is positive.
- **ConcaveTrading**:  
  sell a stock if its return is positive; buy it with available system cash if its return is negative.
- **Buy_Hold**:  
  buy once and hold.

## Important note
This is a **Python approximation** of the SAS logic, not a byte-for-byte `PROC RISK` equivalent.
The original SAS code relies on SAS Risk Dimensions objects such as instrument classes, cash transfer
functions, and project/simulation definitions. Here, those ideas are mapped into a transparent
daily backtest using `pandas`.

## Source reference

Original GitHub file used for the conversion:

`https://github.com/Py14aK/Py14aK/blob/main/Risk/Creating%20accounts`

Key ideas extracted from the source:

- a shared **system cash** account
- per-instrument buy/sell decisions based on **current return vs previous value**
- separate **convex**, **concave**, and **buy-and-hold** projects

In [ ]:
# Install dependencies if needed
# Uncomment the next line the first time you run this notebook.
# !pip install yfinance pandas numpy matplotlib

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError(
        "yfinance is required. Uncomment the pip install line in the setup cell and run it first."
    ) from e

pd.set_option("display.max_columns", 100)

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
tickers = [
    "BYND", "IBM", "PLAB", "AKAM", "TJX",
    "GRAB", "NTCO", "NUE", "RTX", "HWM", "CRS"
]

# If you intended CRISP -> CRSP, add "CRSP" here manually.
# tickers.append("CRSP")

start_date = "2023-01-01"
end_date = None                   # None = up to today
initial_system_cash = 100_000.0  # maps to the SAS shared system cash idea
allow_fractional_shares = False  # SAS uses floor(...) so default is False
plot_log_scale = False

In [ ]:
# -----------------------------
# Download adjusted close prices
# -----------------------------
raw = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False,
)

if raw.empty:
    raise ValueError("No data was downloaded. Check ticker symbols or internet access.")

if isinstance(raw.columns, pd.MultiIndex):
    if "Close" in raw.columns.get_level_values(0):
        prices = raw["Close"].copy()
    else:
        prices = raw.xs("Close", axis=1, level=0).copy()
else:
    prices = raw.copy()

prices = prices.dropna(axis=1, how="all").dropna(axis=0, how="all")
prices = prices.ffill().dropna(how="all")

if prices.empty:
    raise ValueError("Price table is empty after cleaning.")

valid_tickers = list(prices.columns)
print("Tickers used:", valid_tickers)
prices.tail()

In [ ]:
# Daily returns used to mimic the SAS comparison:
# equity_return = (current_value - previous_value) / previous_value
returns = prices.pct_change()
returns.head()

In [ ]:
def _share_count(cash: float, price: float, allow_fractional: bool = False) -> float:
    if price <= 0 or not np.isfinite(price):
        return 0.0
    qty = cash / price
    if allow_fractional:
        return max(qty, 0.0)
    return float(math.floor(qty))


def run_strategy(
    prices: pd.DataFrame,
    strategy: str,
    initial_system_cash: float = 100000.0,
    allow_fractional: bool = False,
):
    '''
    Python approximation of the SAS methods:

    - convex: if return < 0 -> sell all; if return > 0 -> buy with available system cash
    - concave: if return > 0 -> sell all; if return < 0 -> buy with available system cash
    - buy_hold: invest equally on the first available date and hold
    '''
    strategy = strategy.lower().strip()
    if strategy not in {"convex", "concave", "buy_hold"}:
        raise ValueError("strategy must be one of: convex, concave, buy_hold")

    returns = prices.pct_change()
    symbols = list(prices.columns)

    holdings = {s: 0.0 for s in symbols}
    system_cash = float(initial_system_cash)

    history = []

    first_date = prices.index[0]
    first_prices = prices.loc[first_date]

    if strategy == "buy_hold":
        active = [s for s in symbols if pd.notna(first_prices[s]) and first_prices[s] > 0]
        if active:
            cash_per_symbol = system_cash / len(active)
            spent = 0.0
            for s in active:
                qty = _share_count(cash_per_symbol, first_prices[s], allow_fractional)
                holdings[s] = qty
                spent += qty * first_prices[s]
            system_cash -= spent

    for dt in prices.index:
        row_prices = prices.loc[dt]
        row_returns = returns.loc[dt]

        if strategy in {"convex", "concave"} and dt != first_date:
            for s in symbols:
                price = row_prices[s]
                r = row_returns[s]

                if pd.isna(price) or price <= 0 or pd.isna(r):
                    continue

                current_holding = holdings[s]

                sell_signal = (
                    (strategy == "convex" and r < 0) or
                    (strategy == "concave" and r > 0)
                )
                buy_signal = (
                    (strategy == "convex" and r > 0) or
                    (strategy == "concave" and r < 0)
                )

                # Step 1: if signal says sell, liquidate current holding into system cash
                if sell_signal and current_holding > 0:
                    proceeds = current_holding * price
                    system_cash += proceeds
                    holdings[s] = 0.0

                # Step 2: if signal says buy, deploy available system cash into this instrument
                elif buy_signal and system_cash > 0:
                    qty = _share_count(system_cash, price, allow_fractional)
                    if qty > 0:
                        cost = qty * price
                        holdings[s] += qty
                        system_cash -= cost

        equity_value = 0.0
        for s in symbols:
            price = row_prices[s]
            if pd.notna(price):
                equity_value += holdings[s] * price

        total_value = system_cash + equity_value

        history.append({
            "Date": dt,
            "SystemCash": system_cash,
            "EquityValue": equity_value,
            "TotalValue": total_value,
            **{f"Holding_{s}": holdings[s] for s in symbols}
        })

    out = pd.DataFrame(history).set_index("Date")
    return out

In [ ]:
convex = run_strategy(
    prices,
    strategy="convex",
    initial_system_cash=initial_system_cash,
    allow_fractional=allow_fractional_shares,
)

concave = run_strategy(
    prices,
    strategy="concave",
    initial_system_cash=initial_system_cash,
    allow_fractional=allow_fractional_shares,
)

buy_hold = run_strategy(
    prices,
    strategy="buy_hold",
    initial_system_cash=initial_system_cash,
    allow_fractional=allow_fractional_shares,
)

convex.tail()

In [ ]:
# Combine results for comparison
portfolio_values = pd.DataFrame({
    "ConvexTrading": convex["TotalValue"],
    "ConcaveTrading": concave["TotalValue"],
    "BuyHold": buy_hold["TotalValue"],
})

portfolio_values.tail()

In [ ]:
def summary_stats(series: pd.Series) -> pd.Series:
    daily_ret = series.pct_change().dropna()
    total_return = series.iloc[-1] / series.iloc[0] - 1
    ann_return = (1 + total_return) ** (252 / max(len(daily_ret), 1)) - 1 if len(daily_ret) > 0 else np.nan
    ann_vol = daily_ret.std() * np.sqrt(252) if len(daily_ret) > 1 else np.nan
    sharpe = ann_return / ann_vol if ann_vol and ann_vol > 0 else np.nan
    running_max = series.cummax()
    drawdown = series / running_max - 1
    max_dd = drawdown.min()

    return pd.Series({
        "Start": series.index[0],
        "End": series.index[-1],
        "StartValue": series.iloc[0],
        "EndValue": series.iloc[-1],
        "TotalReturn": total_return,
        "AnnualizedReturn": ann_return,
        "AnnualizedVolatility": ann_vol,
        "SharpeApprox": sharpe,
        "MaxDrawdown": max_dd,
    })

stats = pd.DataFrame({name: summary_stats(portfolio_values[name]) for name in portfolio_values.columns}).T
stats

In [ ]:
plt.figure(figsize=(12, 6))
for col in portfolio_values.columns:
    plt.plot(portfolio_values.index, portfolio_values[col], label=col)

if plot_log_scale:
    plt.yscale("log")

plt.title("Portfolio value comparison")
plt.xlabel("Date")
plt.ylabel("Portfolio value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Optional: inspect final holdings
final_holdings = pd.DataFrame({
    "ConvexTrading": convex.filter(like="Holding_").iloc[-1],
    "ConcaveTrading": concave.filter(like="Holding_").iloc[-1],
    "BuyHold": buy_hold.filter(like="Holding_").iloc[-1],
})
final_holdings.index = final_holdings.index.str.replace("Holding_", "", regex=False)
final_holdings

## How this maps to the SAS file

### SAS idea
The original file creates:

- instrument metadata
- a shared `system_cash_01`
- an `EquityPrice` method
- two trading methods:
  - `ConvexTrading`
  - `ConcaveTrading`
- a `Buy_Hold` project

### Python mapping used here
- `system_cash_01` → `system_cash`
- `TRADE(...)` → direct share updates in `holdings`
- `TRANSFERCASH(...)` → increase/decrease of `system_cash`
- `GET_CURRENTVALUE(index, -1, 1)` → previous daily close
- `GET_CURRENTVALUE(index, 0, 1)` → current daily close
- `floor(avail_cash / val_current)` → integer shares only by default

## Differences from SAS Risk Dimensions
This notebook does **not** reproduce:

- SAS environment/project objects
- covariance simulation objects
- crossclass/project/report internals
- exact SAS instrument/account semantics

It does reproduce the **core trading decision logic** in a practical Python form.

## Next modifications you may want
- replace `yfinance` with your own CSV price file
- add transaction costs and slippage
- add a benchmark such as SPY
- change execution to weekly instead of daily
- run the same logic on your custom ticker universe